# Movie Review Classifier
This notebook builds a 3-class text classifier to identify whether a piece of text is:
- Label 0: Not a movie/TV show review
- Label 1: A positive movie/TV show review
- Label 2: A negative movie/TV show review

I use TF-IDF features with Logistic Regression, evaluated using stratified k-fold cross validation and macro F1 score.

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline

## 1. Data Loading and Preprocessing

Here I load the training and test sets from CSV files. Any rows with missing TEXT values are dropped from the training set, and missing values in the test set are replaced with empty strings.

In [2]:
class TextPreprocessor:
    """handles loading and cleaning the data"""
    
    def __init__(self, train_path, test_path):
        self.train_path = train_path
        self.test_path = test_path
    
    def load(self):
        train = pd.read_csv(self.train_path)
        test = pd.read_csv(self.test_path)
        
        train = train.dropna(subset=["TEXT"])  # drop rows with missing text
        test = test.fillna({"TEXT": ""})        # fill missing test text with empty string
        
        return train, test


# load data
preprocessor = TextPreprocessor("../train.csv", "../test.csv")
train, test = preprocessor.load()

X_train = train["TEXT"]
y_train = train["LABEL"]
X_test = test["TEXT"]

print("Train size:", len(train))
print("Test size:", len(test))
print("Class distribution:\n", y_train.value_counts())

Train size: 70310
Test size: 17580
Class distribution:
 LABEL
0    32282
1    19139
2    18889
Name: count, dtype: int64


## 2. Feature Extraction

Here I convert raw text into numerical features using TF-IDF (Term Frequency-Inverse Document Frequency). We use both unigrams and bigrams (`ngram_range=(1, 2)`) to capture individual words as well as two-word phrases, which helps the model pick up on sentiment patterns like "not good" or "highly recommended".

In [3]:
class FeatureExtractor:
    """converts raw text into TF-IDF features"""
    
    def __init__(self, ngram_range=(1, 2), max_features=50000):
        self.vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)
    
    def fit(self, texts):
        self.vectorizer.fit(texts)
    
    def transform(self, texts):
        return self.vectorizer.transform(texts)
    
    def fit_transform(self, texts):
        return self.vectorizer.fit_transform(texts)

## 3. Classifier

I used Logistic Regression as the classification algorithm. Key settings:
- `max_iter=1000` to ensure the model converges on this vocabulary size
- `class_weight="balanced"` to account for the class imbalance we observed in EDA (class 0 is roughly twice the size of classes 1 and 2)
- `random_state=66` for reproducibility

I also evaluate using 5-fold stratified cross validation and report the macro F1 score, which weights all three classes equally regardless of their size.

In [4]:
class MovieReviewClassifier:
    """wraps logistic regression for 3-class classification"""
    
    def __init__(self):
        self.clf = LogisticRegression(
            max_iter=1000,
            class_weight="balanced",  # accounts for class imbalance
            random_state=66
        )
    
    def train(self, features, labels):
        self.clf.fit(features, labels)
    
    def predict(self, features):
        return self.clf.predict(features)
    
    def evaluate(self, X, y, cv=5):
        # build a pipeline for clean cross validation
        pipeline = Pipeline([
            ("tfidf", TfidfVectorizer(ngram_range=(1, 2), max_features=50000)),
            ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))
        ])
        skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)
        scores = cross_val_score(pipeline, X, y, cv=skf, scoring="f1_macro")
        return scores

## 4. Cross Validation

Before training on the full dataset, I will evaluate the model using 5-fold stratified cross validation. This gives me a reliable estimate of how well the model generalizes and helps detect any high variance issues.

In [5]:
classifier = MovieReviewClassifier()
scores = classifier.evaluate(X_train, y_train)

print("CV Macro F1 scores:", scores)
print("Mean:", scores.mean().round(4))
print("Std:", scores.std().round(4))

CV Macro F1 scores: [0.91624267 0.91922152 0.92124257 0.9218782  0.92010186]
Mean: 0.9197
Std: 0.002


### Cross Validation Results

I evaluated the Logistic Regression model using 5-fold stratified cross validation and got a mean macro F1 of **0.9197** with a standard deviation of **0.0020**. The scores are pretty consistent across all five folds, ranging from 0.9162 to 0.9219, which tells me the model isn't overfitting to any particular split of the data. I used `class_weight="balanced"` to handle the class imbalance I noticed in EDA, where class 0 made up about 46% of the training data. Without this setting, the model would likely be biased toward predicting class 0 more often.

I chose Logistic Regression because it's a strong baseline for text classification tasks and is well documented for use with TF-IDF features. You can read more about it in the sklearn LogisticRegression documentation: https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html and about stratified k-fold cross validation in the sklearn model evaluation guide:https://scikit-learn.org/stable/modules/cross_validation.html#stratified-k-fold.

## 5. Training and Prediction

Now we train the classifier on the full training set and generate predictions for the test set.

In [6]:
# fit feature extractor on training data
extractor = FeatureExtractor()
X_train_features = extractor.fit_transform(X_train)
X_test_features = extractor.transform(X_test)

# train classifier on full training data
classifier.train(X_train_features, y_train)

# generate predictions on test set
predictions = classifier.predict(X_test_features)

print("Prediction distribution:")
print(pd.Series(predictions).value_counts())

Prediction distribution:
0    8051
1    4879
2    4650
Name: count, dtype: int64


### Prediction Distribution

The model predicted 8,051 instances as class 0 (not a review), 4,879 as class 1 (positive), and 4,650 as class 2 (negative). This distribution is very similar to what I saw in the training data during EDA, which is reassuring -- the model isn't collapsing into predicting one class overwhelmingly. The `class_weight="balanced"` parameter in Logistic Regression adjusts the weights inversely proportional to class frequencies, which helps prevent the majority class from dominating predictions. More details on handling imbalanced classes can be found in the sklearn documentation on imbalanced datasets: https://scikit-learn.org/stable/modules/svm.html#unbalanced-problems and this guide on class weights in Logistic Regression: https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html#sklearn.linear_model.LogisticRegression.

## 6. Generate Submission File

Formatting the predictions into the required submission format (`ID` and `LABEL` columns) and save to a CSV file for upload to Kaggle.

In [7]:
submission = pd.DataFrame({
    "ID": test["ID"],
    "LABEL": predictions
})

submission.to_csv("../submission_lr.csv", index=False)
print("Submission saved to submission_lr.csv!")
print(submission.head())

Submission saved to submission_lr.csv!
                     ID  LABEL
0   1087873697464833975      1
1   5853461517618045821      1
2   1225516091890726770      2
3  11795874926011119457      0
4  15956464546963841646      0
